In [ ]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import chi2, genpareto
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model

warnings.filterwarnings("ignore")

@dataclass
class Cfg:
    path: str = r"C:\Users\15611\GPD\Data\sp500-dataset.csv"
    out: str = "out_sp500_evt"
    r: str = "SP500_return"
    l: str = "SP500_loss"
    x: str = "VIX_lag1"
    n: int = 500
    w: int = 500
    ar: int = 1
    ma: int = 1
    u: float = 0.90
    a: tuple[float, ...] = (0.95, 0.99)
    vix: bool = True
    reg: bool = True
    fig: bool = True

cfg = Cfg()

mdl = ["st", "g", "ag", "aag", "afi", "vx_aag", "vx_afi"]

mname = {
    "st": "Static-GPD",
    "g": "GARCH-EVT",
    "ag": "ARMA-GARCH-EVT",
    "aag": "ARMA-AGARCH-EVT",
    "afi": "ARMA-FIGARCH-EVT",
    "vx_aag": "VIX-conditioned ARMA-AGARCH-GPD",
    "vx_afi": "VIX-conditioned ARMA-FIGARCH-GPD",
}

pname = {"gfc": "GFC period", "covid": "Covid period", "recent": "Recent period", "all": "All periods combined"}

def lab(a):
    return str(int(round(a * 100))) if abs(a * 100 - round(a * 100)) < 1e-8 else str(int(round(a * 1000)))

def load(c):
    cols = [c.r, c.l] + ([c.x] if c.x else [])
    z = pd.read_csv(c.path, index_col=0, parse_dates=True).sort_index()
    miss = [k for k in cols if k not in z.columns]
    if miss:
        raise ValueError(f"missing columns: {miss}")
    z = z[cols].dropna()
    print(z.index.min().date(), z.index.max().date(), len(z))
    return z

def gp_var(a, u, pu, xi, b):
    if b <= 0 or pu <= 0:
        return np.nan
    if abs(xi) < 1e-8:
        return u + b * np.log(pu / (1 - a))
    return u + (b / xi) * (((1 - a) / pu) ** (-xi) - 1)

def gp_es(a, u, pu, xi, b):
    q = gp_var(a, u, pu, xi, b)
    if np.isnan(q) or xi >= 1:
        return np.nan
    if abs(xi) < 1e-8:
        return q + b
    return (q + b - xi * u) / (1 - xi)

def gp_fit(s, q):
    s = s.dropna()
    u = s.quantile(q)
    e = s[s > u] - u
    pu = (s > u).mean()
    if len(e) < 10:
        raise RuntimeError(f"few exceedances: {len(e)}")
    xi, _, b = genpareto.fit(e, floc=0)
    return {"u": float(u), "pu": float(pu), "xi": float(xi), "b": float(b), "ne": int(len(e))}

def kuc(v, a):
    v = np.asarray(v).astype(int)
    n = len(v)
    x = int(v.sum())
    if n == 0:
        return np.nan, np.nan
    p = 1 - a
    ph = np.clip(x / n, 1e-12, 1 - 1e-12)
    l0 = (n - x) * np.log(1 - p) + x * np.log(p)
    l1 = (n - x) * np.log(1 - ph) + x * np.log(ph)
    lr = -2 * (l0 - l1)
    return float(lr), float(1 - chi2.cdf(lr, 1))

def ind(v):
    v = np.asarray(v).astype(int)
    if len(v) < 2:
        return np.nan, np.nan
    n00 = n01 = n10 = n11 = 0
    for i in range(1, len(v)):
        if v[i - 1] == 0 and v[i] == 0:
            n00 += 1
        elif v[i - 1] == 0 and v[i] == 1:
            n01 += 1
        elif v[i - 1] == 1 and v[i] == 0:
            n10 += 1
        else:
            n11 += 1
    nt = n00 + n01 + n10 + n11
    if nt == 0:
        return np.nan, np.nan
    p = np.clip((n01 + n11) / nt, 1e-12, 1 - 1e-12)
    p01 = np.clip(n01 / (n00 + n01), 1e-12, 1 - 1e-12) if n00 + n01 else 1e-12
    p11 = np.clip(n11 / (n10 + n11), 1e-12, 1 - 1e-12) if n10 + n11 else 1e-12
    l0 = (n00 + n10) * np.log(1 - p) + (n01 + n11) * np.log(p)
    l1 = n00 * np.log(1 - p01) + n01 * np.log(p01) + n10 * np.log(1 - p11) + n11 * np.log(p11)
    lr = -2 * (l0 - l1)
    return float(lr), float(1 - chi2.cdf(lr, 1))

def cc(v, a):
    x, _ = kuc(v, a)
    y, _ = ind(v)
    z = x + y
    return float(z), float(1 - chi2.cdf(z, 2))

def qloss(y, q, a):
    z = pd.concat([y.rename("y"), q.rename("q")], axis=1).dropna()
    yy = z["y"].to_numpy()
    qq = z["q"].to_numpy()
    return float(np.mean((a - (yy < qq).astype(int)) * (yy - qq)))

def eratio(y, q, e):
    z = pd.concat([y.rename("y"), q.rename("q"), e.rename("e")], axis=1).dropna()
    m = z["y"] > z["q"]
    if m.sum() == 0:
        return np.nan
    return float(z.loc[m, "y"].mean() / z.loc[m, "e"].mean())

def eabs(y, q, e):
    z = pd.concat([y.rename("y"), q.rename("q"), e.rename("e")], axis=1).dropna()
    m = z["y"] > z["q"]
    if m.sum() == 0:
        return np.nan
    return float(np.abs(z.loc[m, "y"] - z.loc[m, "e"]).mean())

def ev(bt, c, m, p, rg=None):
    out = []
    for a in c.a:
        la = lab(a)
        qc = f"VaR_{la}_{m}"
        ec = f"ES_{la}_{m}"
        if qc not in bt.columns:
            continue
        z = bt[[c.l, qc]].dropna()
        if len(z) == 0:
            continue
        y = z[c.l]
        q = z[qc]
        vv = (y > q).astype(int)
        _, pk = kuc(vv, a)
        _, pi = ind(vv)
        _, pc = cc(vv, a)
        row = {"period": p, "model": m, "level": a, "n": int(len(vv)), "viol": int(vv.sum()), "vr": float(vv.mean()), "exp": 1 - a, "kupiec": pk, "ind": pi, "cc": pc, "qloss": qloss(y, q, a)}
        if rg is not None:
            row["regime"] = rg
        if ec in bt.columns:
            ee = bt.loc[z.index, ec]
            row["esr"] = eratio(y, q, ee)
            row["eae"] = eabs(y, q, ee)
        out.append(row)
    return pd.DataFrame(out)

def periods(df, c):
    idx = df.index
    def take(d, nm):
        s0 = idx[idx >= d]
        if len(s0) == 0:
            raise ValueError(f"no data after {d}")
        i = idx.get_loc(s0[0])
        if i < c.w:
            raise ValueError(f"{nm}: only {i} obs before first test day; need {c.w}")
        if i + c.n > len(idx):
            raise ValueError(f"{nm}: not enough test obs")
        return pd.DatetimeIndex(idx[i:i + c.n])
    ans = {"gfc": take("2008-01-01", "gfc"), "covid": take("2020-01-01", "covid"), "recent": pd.DatetimeIndex(idx[-c.n:])}
    rows = []
    for k, v in ans.items():
        rows.append({"period": k, "start": v[0], "end": v[-1], "n": len(v), "w": c.w})
        print(k, v[0].date(), v[-1].date(), len(v), c.w)
    pd.DataFrame(rows).to_csv(os.path.join(c.out, "periods.csv"), index=False)
    return ans

def arma(y, c):
    try:
        f = ARIMA(y, order=(c.ar, 0, c.ma), trend="c").fit()
        mu = f.fittedvalues.reindex(y.index)
        mn = float(f.get_forecast(steps=1).predicted_mean.iloc[0])
        return mu, mn
    except Exception:
        m = float(y.mean())
        return pd.Series(m, index=y.index), m

def vol_fit(tr, c, m):
    y = tr[c.r] * 100
    if m == "g":
        am = arch_model(y, mean="Constant", vol="GARCH", p=1, q=1, dist="t", rescale=False)
        f = am.fit(disp="off")
        fc = f.forecast(horizon=1, reindex=False)
        mn = float(fc.mean.iloc[-1, 0])
        sn = float(np.sqrt(fc.variance.iloc[-1, 0]))
        zr = f.std_resid.dropna()
    else:
        mu, mn = arma(y, c)
        e = (y - mu).dropna()
        if m == "ag":
            am = arch_model(e, mean="Zero", vol="GARCH", p=1, q=1, dist="t", rescale=False)
        elif m == "aag":
            am = arch_model(e, mean="Zero", vol="GARCH", p=1, o=1, q=1, dist="t", rescale=False)
        elif m == "afi":
            am = arch_model(e, mean="Zero", vol="FIGARCH", p=1, q=1, dist="t", rescale=False)
        else:
            raise ValueError(m)
        f = am.fit(disp="off")
        fc = f.forecast(horizon=1, reindex=False)
        sn = float(np.sqrt(fc.variance.iloc[-1, 0]))
        zr = (f.resid / f.conditional_volatility).dropna()
    z = -zr
    ta = gp_fit(z, c.u)
    zv = {a: gp_var(a, ta["u"], ta["pu"], ta["xi"], ta["b"]) for a in c.a}
    ze = {a: gp_es(a, ta["u"], ta["pu"], ta["xi"], ta["b"]) for a in c.a}
    return {"mu": mn, "sig": sn, "zv": zv, "ze": ze, "ta": ta, "fit": f, "zr": zr}

def cnll(par, y, x):
    xi, b0, b1 = par
    b = np.exp(b0 + b1 * x)
    if np.any(~np.isfinite(b)) or np.any(b <= 0) or xi >= 1:
        return 1e12
    if abs(xi) < 1e-7:
        return float(np.sum(np.log(b) + y / b))
    a = 1 + xi * y / b
    if np.any(a <= 0):
        return 1e12
    return float(np.sum(np.log(b) + (1 / xi + 1) * np.log(a)))

def cgp_fit(z, x, uq):
    dat = pd.concat([z.rename("z"), x.rename("x")], axis=1).dropna()
    u = dat["z"].quantile(uq)
    ms = dat["z"] > u
    ex = dat.loc[ms]
    if len(ex) < 10:
        raise RuntimeError("few conditional exceedances")
    xm = float(dat["x"].mean())
    xs = float(dat["x"].std())
    if xs == 0 or np.isnan(xs):
        raise RuntimeError("bad x scale")
    y = (ex["z"] - u).to_numpy()
    xx = ((ex["x"] - xm) / xs).to_numpy()
    xi0, _, b0 = genpareto.fit(y, floc=0)
    ini = np.array([xi0, np.log(max(b0, 1e-6)), 0.0])
    bds = [(-0.5, 0.95), (None, None), (-5, 5)]
    r = minimize(cnll, ini, args=(y, xx), method="L-BFGS-B", bounds=bds)
    if not r.success:
        ini = np.array([0.05, np.log(max(np.std(y), 1e-6)), 0.0])
        r = minimize(cnll, ini, args=(y, xx), method="L-BFGS-B", bounds=bds)
    if not r.success:
        raise RuntimeError(r.message)
    xi, a0, a1 = r.x
    return {"u": float(u), "pu": float(ms.mean()), "xi": float(xi), "a0": float(a0), "a1": float(a1), "xm": xm, "xs": xs, "ne": int(len(ex))}

def vx_fit(tr, c, m, xn):
    base = "aag" if m == "vx_aag" else "afi"
    f = vol_fit(tr, c, base)
    z = -f["zr"]
    x = tr[c.x].reindex(z.index)
    ta = cgp_fit(z, x, c.u)
    xx = (float(xn) - ta["xm"]) / ta["xs"]
    bn = float(np.exp(ta["a0"] + ta["a1"] * xx))
    f["zv"] = {a: gp_var(a, ta["u"], ta["pu"], ta["xi"], bn) for a in c.a}
    f["ze"] = {a: gp_es(a, ta["u"], ta["pu"], ta["xi"], bn) for a in c.a}
    f["ta"] = ta
    f["bn"] = bn
    return f

def fit_m(tr, c, m, xn=None):
    if m == "st":
        ta = gp_fit(tr[c.l], c.u)
        rv = {a: gp_var(a, ta["u"], ta["pu"], ta["xi"], ta["b"]) for a in c.a}
        re = {a: gp_es(a, ta["u"], ta["pu"], ta["xi"], ta["b"]) for a in c.a}
        return {"rv": rv, "re": re, "ta": ta}
    if m in ["g", "ag", "aag", "afi"]:
        return vol_fit(tr, c, m)
    if m in ["vx_aag", "vx_afi"]:
        return vx_fit(tr, c, m, xn)
    raise ValueError(m)

def roll(df, c, td, nm, mods=None):
    if mods is None:
        mods = mdl.copy()
    if not c.vix:
        mods = [m for m in mods if not m.startswith("vx_")]
    rows = []
    for j, d in enumerate(td, 1):
        pos = df.index.get_loc(d)
        tr = df.iloc[pos - c.w:pos].copy()
        row = {"date": d, c.r: df.loc[d, c.r], c.l: df.loc[d, c.l]}
        if c.x in df.columns:
            row[c.x] = df.loc[d, c.x]
        xn = df.loc[d, c.x] if c.x in df.columns else None
        print(nm, d.date(), j, len(td))
        for m in mods:
            try:
                f = fit_m(tr, c, m, xn)
                if m == "st":
                    for a in c.a:
                        la = lab(a)
                        row[f"VaR_{la}_{m}"] = f["rv"][a]
                        row[f"ES_{la}_{m}"] = f["re"][a]
                else:
                    for a in c.a:
                        la = lab(a)
                        row[f"VaR_{la}_{m}"] = (-f["mu"] + f["sig"] * f["zv"][a]) / 100
                        row[f"ES_{la}_{m}"] = (-f["mu"] + f["sig"] * f["ze"][a]) / 100
                ta = f["ta"]
                row[f"xi_{m}"] = ta.get("xi", np.nan)
                row[f"ne_{m}"] = ta.get("ne", np.nan)
                row[f"u_{m}"] = ta.get("u", np.nan)
                if "a1" in ta:
                    row[f"b1_{m}"] = ta["a1"]
                    row[f"bn_{m}"] = f.get("bn", np.nan)
                if m in ["afi", "vx_afi"]:
                    for k, v in f["fit"].params.items():
                        if k.lower() == "d":
                            row[f"d_{m}"] = v
            except Exception as e:
                print("fail", m, d.date(), e)
                for a in c.a:
                    la = lab(a)
                    row[f"VaR_{la}_{m}"] = np.nan
                    row[f"ES_{la}_{m}"] = np.nan
        rows.append(row)
    bt = pd.DataFrame(rows).set_index("date")
    mets = []
    for m in mods:
        z = ev(bt, c, m, nm)
        if len(z):
            mets.append(z)
    mets = pd.concat(mets, ignore_index=True) if mets else pd.DataFrame()
    return bt, mets

def add_reg(bt, c):
    z = bt.copy()
    if c.x not in z.columns:
        raise ValueError(f"{c.x} not in bt")
    x = z[c.x].dropna()
    q50 = x.quantile(0.50)
    q75 = x.quantile(0.75)
    q90 = x.quantile(0.90)
    z["normal"] = z[c.x] <= q50
    z["stress"] = z[c.x] > q75
    z["crisis"] = z[c.x] > q90
    z["x50"] = q50
    z["x75"] = q75
    z["x90"] = q90
    return z

def reg_eval(bts, c, mods=None):
    if mods is None:
        mods = mdl.copy()
    if not c.vix:
        mods = [m for m in mods if not m.startswith("vx_")]
    out = []
    for p, bt in bts.items():
        z = add_reg(bt, c)
        rgs = {"normal": z["normal"], "stress": z["stress"], "crisis": z["crisis"]}
        print("reg", p, z.index.min().date(), z.index.max().date(), z["x50"].iloc[0], z["x75"].iloc[0], z["x90"].iloc[0])
        for r, ms in rgs.items():
            sub = z.loc[ms]
            print(r, len(sub))
            for m in mods:
                e = ev(sub, c, m, p, r)
                if len(e):
                    out.append(e)
    if not out:
        return pd.DataFrame()
    ans = pd.concat(out, ignore_index=True)
    ans.to_csv(os.path.join(c.out, "regime.csv"), index=False)
    return ans

def vplot(bt, c, p, a=0.99):
    if not c.fig:
        return
    la = lab(a)
    plt.figure(figsize=(13, 5))
    plt.plot(bt.index, bt[c.l], label="loss", lw=0.8)
    for m in mdl:
        col = f"VaR_{la}_{m}"
        if col in bt.columns:
            plt.plot(bt.index, bt[col], lw=1.0, label=mname[m])
    plt.axhline(0, lw=0.7)
    plt.title(f"{pname.get(p, p)} {int(a * 100)}% VaR")
    plt.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(c.out, f"{p}_var_{la}.png"), dpi=250, bbox_inches="tight")
    plt.show()

def main(c=cfg):
    os.makedirs(c.out, exist_ok=True)
    df = load(c)
    ps = periods(df, c)
    bts = {}
    mets = []
    for p, td in ps.items():
        print("run", p, td[0].date(), td[-1].date(), len(td), c.w)
        bt, mt = roll(df, c, td, p)
        bts[p] = bt
        mets.append(mt)
        bt.to_csv(os.path.join(c.out, f"{p}_bt.csv"))
        bt.to_csv(os.path.join(c.out, f"{p}_all_models_backtest.csv"))
        vplot(bt, c, p, 0.99)
    met = pd.concat(mets, ignore_index=True)
    met.to_csv(os.path.join(c.out, "metrics.csv"), index=False)
    met.to_csv(os.path.join(c.out, "three_period_all_models_metrics.csv"), index=False)
    rg = reg_eval(bts, c) if c.reg else None
    return bts, met, rg

def nice(x):
    z = x.copy()
    if "model" in z.columns:
        z["model"] = z["model"].replace(mname)
    if "period" in z.columns:
        z["period"] = z["period"].replace(pname)
    cols = ["period", "regime", "model", "level", "n", "viol", "vr", "exp", "kupiec", "ind", "cc", "qloss", "esr", "eae"]
    z = z[[k for k in cols if k in z.columns]]
    for k in z.columns:
        if k not in ["period", "regime", "model"]:
            try:
                z[k] = z[k].round(4)
            except Exception:
                pass
    return z

def overall(bts, c):
    z = []
    for p, bt in bts.items():
        u = bt.copy()
        u["src"] = p
        z.append(u)
    z = pd.concat(z).sort_index()
    out = []
    for m in mdl:
        e = ev(z, c, m, "all")
        if len(e):
            out.append(e)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

def tabs(bts, met, rg, c=cfg):
    os.makedirs(c.out, exist_ok=True)
    ptab = nice(met)
    otab = nice(overall(bts, c))
    ftab = pd.concat([ptab, otab], ignore_index=True)
    rtab = nice(rg) if rg is not None and len(rg) else pd.DataFrame()
    ptab.to_csv(os.path.join(c.out, "tab_period.csv"), index=False)
    otab.to_csv(os.path.join(c.out, "tab_overall.csv"), index=False)
    ftab.to_csv(os.path.join(c.out, "tab_full.csv"), index=False)
    if len(rtab):
        rtab.to_csv(os.path.join(c.out, "tab_regime.csv"), index=False)
    xls = os.path.join(c.out, "final_tables.xlsx")
    with pd.ExcelWriter(xls, engine="openpyxl") as w:
        ptab.to_excel(w, "period", index=False)
        otab.to_excel(w, "overall", index=False)
        ftab.to_excel(w, "full", index=False)
        if len(rtab):
            rtab.to_excel(w, "regime", index=False)
        for p in ptab["period"].unique():
            ptab[ptab["period"] == p].to_excel(w, str(p)[:31], index=False)
    print(xls)
    try:
        display(ptab)
        display(otab)
        if len(rtab):
            display(rtab)
    except NameError:
        print(ptab)
        print(otab)
        if len(rtab):
            print(rtab)
    return {"period": ptab, "overall": otab, "full": ftab, "regime": rtab}

def load_old(c=cfg):
    bts = {}
    files = {
        "gfc": ["gfc_bt.csv", "GFC_from_2008_500d_all_models_backtest.csv"],
        "covid": ["covid_bt.csv", "Covid_from_2020_500d_all_models_backtest.csv"],
        "recent": ["recent_bt.csv", "Recent_500d_all_models_backtest.csv"],
    }
    for p, fs in files.items():
        for f in fs:
            path = os.path.join(c.out, f)
            if os.path.exists(path):
                bts[p] = pd.read_csv(path, index_col=0, parse_dates=True)
                print("load", path, bts[p].shape)
                break
    mp = os.path.join(c.out, "metrics.csv")
    if not os.path.exists(mp):
        mp = os.path.join(c.out, "three_period_all_models_metrics.csv")
    if os.path.exists(mp):
        met = pd.read_csv(mp)
    else:
        out = []
        for p, bt in bts.items():
            for m in mdl:
                e = ev(bt, c, m, p)
                if len(e):
                    out.append(e)
        met = pd.concat(out, ignore_index=True)
    rp = os.path.join(c.out, "regime.csv")
    if os.path.exists(rp):
        rg = pd.read_csv(rp)
    else:
        rg = reg_eval(bts, c) if c.reg else None
    return bts, met, rg

if __name__ == "__main__":
    bts, met, rg = main(cfg)
    res = tabs(bts, met, rg, cfg)
